In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    coalesce,
    current_timestamp,
    trim,
    from_utc_timestamp,
    lit,
    sha2,
    upper,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
#dbutils.widgets.text('destination_schema_zone', 'None', '07-Destination Schema Zone') # change for testing
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')
#destination_schema_zone = dbutils.widgets.get('destination_schema_zone')    # change for testing

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_legal_action_relationships_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) : # change for testing

        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "LegalActionRelationshipsId string"
        #     ",SourceSystem string"
        #     ",IngestionTimestamp timestamp"
        #     ",BatchId string"
        #     ",LegalActionRelationshipsKey string"
        #     ",LegalActionRelationshipsTypeCode string"
        #     ",LegalActionRelationshipsType string"
        #     ",LegalActionRelationshipsSubTypeCode string"
        #     ",LegalActionRelationshipsSubType string"
        #     ",LegalActionKey string"
        #     ",RelatedLegalActionKey string"
        #     ",CreationDate date"
        #     ",TerminationDate date"
        #     ",LastUpdateDate date"
        #     ",Comments string"
        #     ",PartyKey string"
        #     ",PartyName string"
        #     ",BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.LegalActionRelationships" # στον πίνακα αυτόν θα γράψει τελικά το script μας # change for testing

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "arotron",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        
        # ================== crmBA ==================

        crmBA_extra_filters = None

        if crmBA_extra_filters is not None:
            crmBA_extra_filters.append(date_filter)
            crmBA_final_filters = " AND ".join(crmBA_extra_filters)
        else:
            crmBA_final_filters = date_filter

        crmBA_selected_fields = [
            "crmBA_ID",
            "crmBE_ID",
            "businessdate",
        ]

        crmBA = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmBA",
            crmBA_final_filters,
            crmBA_selected_fields,
        )
        

        # ================== crmREL ==================

        crmREL_extra_filters = None

        if crmREL_extra_filters is not None:
            crmREL_extra_filters.append(date_filter)
            crmREL_final_filters = " AND ".join(crmREL_extra_filters)
        else:
            crmREL_final_filters = date_filter

        crmREL_selected_fields = [
            "crmREL_ID",
            "crmREL_PR_ID",
            "crmREL_SR_ID",
            "crmSOBJ_PR_ID",
            "crmSOBJ_SR_ID",
            "crmREL_Description",
            "crmREL_Interpretation",
            "crmREL_IsValid",
            "crmREL_ValidFrom",
            "crmREL_ValidTo",
            "businessdate",
        ]

        crmREL = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmREL",
            crmREL_final_filters,
            crmREL_selected_fields,
        )        

        # ================== crmPACT ==================

        crmPACT_extra_filters = None

        if crmPACT_extra_filters is not None:
            crmPACT_extra_filters.append(date_filter)
            crmPACT_final_filters = " AND ".join(crmPACT_extra_filters)
        else:
            crmPACT_final_filters = date_filter

        crmPACT_selected_fields = [
            "crmBA_ID",
            "crmPACT_ActionID",
            "businessdate",
        ]

        crmPACT = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmPACT",
            crmPACT_final_filters,
            crmPACT_selected_fields,
        )


        # ================== crmLOV ==================

        crmLOV_extra_filters = None

        if crmLOV_extra_filters is not None:
            crmLOV_extra_filters.append(date_filter)
            crmLOV_final_filters = " AND ".join(crmLOV_extra_filters)
        else:
            crmLOV_final_filters = date_filter

        crmLOV_selected_fields = [
            "crmLOV_ID",
            "crmLOV_ParentID",
            "businessdate",
        ]

        crmLOV = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmLOV",
            crmLOV_final_filters,
            crmLOV_selected_fields,
        )

        # ================== crmGN ==================

        crmGN_extra_filters = None

        if crmGN_extra_filters is not None:
            crmGN_extra_filters.append(date_filter)
            crmGN_final_filters = " AND ".join(crmGN_extra_filters)
        else:
            crmGN_final_filters = date_filter

        crmGN_selected_fields = [
            "crmGN_ID",
            "crmGN_RowID",
            "crmGN_Comment",
            "crmGN_CreateDT",
            "crmGN_LastUpdDT",
            "crmSOBJ_ID",
            "businessdate",
        ]

        crmGN = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmGN",
            crmGN_final_filters,
            crmGN_selected_fields,
        )

        # ================== crmBE ==================

        crmBE_extra_filters = None

        if crmBE_extra_filters is not None:
            crmBE_extra_filters.append(date_filter)
            crmBE_final_filters = " AND ".join(crmBE_extra_filters)
        else:
            crmBE_final_filters = date_filter

        crmBE_selected_fields = [
            "crmBE_ID",
            "crmCOFF_id",
            "businessdate",
        ]

        crmBE = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmBE",
            crmBE_final_filters,
            crmBE_selected_fields,
        )

        # ================== crmCOFF ==================

        crmCOFF_extra_filters = None

        if crmCOFF_extra_filters is not None:
            crmCOFF_extra_filters.append(date_filter)
            crmCOFF_final_filters = " AND ".join(crmCOFF_extra_filters)
        else:
            crmCOFF_final_filters = date_filter

        crmCOFF_selected_fields = [
            "crmCOFF_id",
            "businessdate",
        ]

        crmCOFF = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmCOFF",
            crmCOFF_final_filters,
            crmCOFF_selected_fields,
        )

        # ================== crmMLT ==================

        crmMLT_extra_filters = None

        if crmMLT_extra_filters is not None:
            crmMLT_extra_filters.append(date_filter)
            crmMLT_final_filters = " AND ".join(crmMLT_extra_filters)
        else:
            crmMLT_final_filters = date_filter

        crmMLT_selected_fields = [
            "crmMLT_ID",
            "crmMLT_LanguageID",
            "crmMLT_ReferenceText",
            "businessdate",
        ]

        crmMLT = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmMLT",
            crmMLT_final_filters,
            crmMLT_selected_fields,
        )

        # ================== crmBP ==================

        crmBP_extra_filters = None
        if crmBP_extra_filters is not None:
            crmBP_extra_filters.append(date_filter)
            crmBP_final_filters = " AND ".join(crmBP_extra_filters)
        else:
            crmBP_final_filters = date_filter
        crmBP_selected_fields = [
            "crmBP_ID",
            "crmBP_Alias",
            "businessdate",
        ]
        crmBP = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmBP",
            crmBP_final_filters,
            crmBP_selected_fields,
        )

        # ================== CrmWvbpInd ==================

        CrmWvbpInd_extra_filters = None
        if CrmWvbpInd_extra_filters is not None:
            CrmWvbpInd_extra_filters.append(date_filter)
            CrmWvbpInd_final_filters = " AND ".join(CrmWvbpInd_extra_filters)
        else:
            CrmWvbpInd_final_filters = date_filter
        CrmWvbpInd_selected_fields = [
            "crmbp_id",
            "crmwvBP_IND_COFFCustCode",
            "businessdate",
        ]
        CrmWvbpInd = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_CrmWvbpInd",
            CrmWvbpInd_final_filters,
            CrmWvbpInd_selected_fields,
        )

        # ================== CrmWvbpOrg ==================

        CrmWvbpOrg_extra_filters = None
        if CrmWvbpOrg_extra_filters is not None:
            CrmWvbpOrg_extra_filters.append(date_filter)
            CrmWvbpOrg_final_filters = " AND ".join(CrmWvbpOrg_extra_filters)
        else:
            CrmWvbpOrg_final_filters = date_filter
        CrmWvbpOrg_selected_fields = [
            "crmbp_id",
            "crmwvBPORG_COFFCustCode",
            "businessdate",
        ]
        CrmWvbpOrg = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_CrmWvbpOrg",
            CrmWvbpOrg_final_filters,
            CrmWvbpOrg_selected_fields,
        )

        # ================== legactrel ==================
        # Preprocess crmLOV – UPPER() + filter on ParentID
        crmLOV_filtered = crmLOV \
            .withColumn("crmLOV_ParentID", upper(crmLOV["crmLOV_ParentID"])) \
            .filter(crmLOV["crmLOV_ParentID"].isin(
                "SYS_BA_LEGAL_ACTION_CODES",
                "SYS_BA_CATEGORY_ADMIN"
            ))

        # Build the "valid SR ids" set, but rename the column so it's NOT crmBA_ID
        subquery_ids = crmBA \
            .join(crmPACT, crmPACT["crmBA_ID"] == crmBA["crmBA_ID"], "inner") \
            .join(crmLOV_filtered, crmLOV_filtered["crmLOV_ID"] == crmPACT["crmPACT_ActionID"], "inner") \
            .select(crmBA["crmBA_ID"].alias("valid_sr_id")) \
            .distinct()
        
        # Join conditions
        crmBA_crmREL_join_condition = crmREL["crmREL_PR_ID"] == crmBA["crmBA_ID"]
        crmREL_subquery_join_condition = crmREL["crmREL_SR_ID"] == subquery_ids["valid_sr_id"]
        crmBA_crmPACT_join_condition = crmPACT["crmBA_ID"] == crmBA["crmBA_ID"]
        crmPACT_crmLOV_join_condition = crmLOV_filtered["crmLOV_ID"] == crmPACT["crmPACT_ActionID"]
        
        # Perform the joins
        legactrel_joined_df = crmBA \
            .join(crmREL, crmBA_crmREL_join_condition, "inner") \
            .join(subquery_ids, crmREL_subquery_join_condition, "left_semi") \
            .join(crmPACT, crmBA_crmPACT_join_condition, "inner") \
            .join(crmLOV_filtered, crmPACT_crmLOV_join_condition, "inner")

        # UPPER() and filter crmREL_Interpretation
        legactrel_filtered_df = legactrel_joined_df \
            .withColumn("crmREL_Interpretation", upper(legactrel_joined_df["crmREL_Interpretation"])) \
            .filter(legactrel_joined_df["crmREL_Interpretation"].isin(
                "SYS_RELATIVE_BA_TYPE_1",
                "SYS_RELATIVE_BA_TYPE_2",
                "SYS_RELATIVE_BA_TYPE_3"
            )) 

        legactrel = legactrel_filtered_df \
            .select(
                legactrel_filtered_df["crmLOV_ID"],
                trim(legactrel_filtered_df["crmREL_ID"].cast("string")).alias("RelationKey"),
                trim(legactrel_filtered_df["crmREL_PR_ID"].cast("string")).alias("PrimaryLegalSourceKey"),
                trim(legactrel_filtered_df["crmREL_SR_ID"].cast("string")).alias("SecondaryLegalSourceKey"),
                trim(legactrel_filtered_df["crmSOBJ_PR_ID"].cast("string")).alias("crmSOBJ_PR_ID"),
                trim(legactrel_filtered_df["crmREL_Description"].cast("string")).alias("RelationDescription"),
                trim(legactrel_filtered_df["crmREL_Interpretation"].cast("string")).alias("RelationInterpretation"),
                legactrel_filtered_df["crmREL_ValidFrom"].alias("RelCreationDate"),
                legactrel_filtered_df["crmREL_ValidTo"].alias("RelterminationDate"),
                crmBA["businessdate"].alias("businessdate")
            ) \
            .distinct()

        # ================== legactcom ==================
        # Legactcom_filtered
        legactcom_filtered = crmGN.filter(
            crmGN["crmSOBJ_ID"] == "SYS_CRM_BA"
        )

        # Legactcom
        legactcom = legactcom_filtered.select(
                    trim(crmGN.crmGN_ID.cast("string")).alias("LegalActComId"),
                    trim(crmGN.crmGN_RowID.cast("string")).alias("LegalActId"),
                    trim(crmGN.crmGN_Comment.cast("string")).alias("Comments"),
                    crmGN.crmGN_CreateDT.alias("CreateDT"),
                    crmGN.crmGN_LastUpdDT.alias("LastUpdDT"),
                    crmGN.crmSOBJ_ID.alias("crmSOBJ_ID"),
                    crmGN.businessdate.alias("BusinessDate")
                )

        # ================== legalactiontoperson ==================

        # Subquery for use in the joins
        subquery_t = crmBA \
            .join(crmBE, crmBA.crmBE_ID == crmBE.crmBE_ID, "inner") \
            .join(crmCOFF, crmBE.crmCOFF_id == crmCOFF.crmCOFF_id, "inner") \
            .join(crmREL, crmREL.crmREL_PR_ID == crmBA.crmBA_ID, "inner") \
            .filter(
                (crmREL.crmSOBJ_PR_ID == 'SYS_CRM_BA') &
                (crmREL.crmSOBJ_SR_ID == 'SYS_CRM_BP')
            ) \
            .select(
                crmREL.crmREL_ID,
                crmREL.crmREL_SR_ID,
                crmREL.crmREL_Interpretation,
                crmBA.crmBA_ID,
                crmREL.crmREL_PR_ID,
                crmREL.crmREL_IsValid,
                crmREL.businessdate
            ).alias("t")

        # Join conditions
        t_crmMLT_join_condition = (
            (crmMLT.crmMLT_ID == subquery_t.crmREL_Interpretation) & 
            (crmMLT.crmMLT_LanguageID == 'SYS_LANG_GR')
        )
        t_crmPACT_join_condition = (crmPACT.crmBA_ID == subquery_t.crmBA_ID)
        crmPACT_crmLOV_join_condition = (crmLOV.crmLOV_ID == crmPACT.crmPACT_ActionID)
        t_crmBP_join_condition = (crmBP.crmBP_ID == subquery_t.crmREL_SR_ID)
        crmBP_CrmWvbpInd_join_condition = (crmBP.crmBP_ID == CrmWvbpInd.crmbp_id)
        crmBP_CrmWvbpOrg_join_condition = (crmBP.crmBP_ID == CrmWvbpOrg.crmbp_id)

        # Joins
        legalactiontoperson_joined_df = subquery_t \
            .join(crmMLT, t_crmMLT_join_condition, "inner") \
            .join(crmPACT, t_crmPACT_join_condition, "inner") \
            .join(crmLOV, crmPACT_crmLOV_join_condition, "inner") \
            .join(crmBP, t_crmBP_join_condition, "inner") \
            .join(CrmWvbpInd, crmBP_CrmWvbpInd_join_condition, "left") \
            .join(CrmWvbpOrg, crmBP_CrmWvbpOrg_join_condition, "left")

        # Legalactiontopersonl_filtered
        legalactiontoperson_filtered = legalactiontoperson_joined_df.filter(
            (crmLOV.crmLOV_ParentID.isin("SYS_BA_LEGAL_ACTION_CODES", "SYS_BA_CATEGORY_ADMIN")) &
            (subquery_t.crmREL_IsValid == 1)
        )

        # Legalactiontoperson
        legalactiontoperson = legalactiontoperson_filtered.select(
            trim(subquery_t.crmREL_ID.cast("string")).alias("RelKey"),
            trim(subquery_t.crmREL_PR_ID.cast("string")).alias("LegalActId"),
            trim(crmBP.crmBP_Alias.cast("string")).alias("AliasSecondary"),
            trim(coalesce(CrmWvbpInd.crmwvBP_IND_COFFCustCode, CrmWvbpOrg.crmwvBPORG_COFFCustCode).cast("string")).alias("CDI"),
            trim(crmMLT.crmMLT_ReferenceText.cast("string")).alias("RelationType"),
            subquery_t["businessdate"].cast("date").alias("BusinessDate")
        ).distinct()

        # =============== Primary Keys - legal_action_relationships_arotron ===============

        # legactrel
        legactrel_primary_key_fields = [
            legactrel.RelationKey,
            lit("LegalAction-LegalAction"),
            lit("arotron"),
        ]

        legactrel = legactrel.withColumn(
            "LegalActionRelationshipsId",
            sha2(concat_ws(";", *legactrel_primary_key_fields), 256),
        )

        # legactcom
        legactcom_primary_key_fields = [
            legactcom.LegalActComId,
            lit("LegalAction-Comments"),
            lit("arotron"),
        ]

        legactcom = legactcom.withColumn(
            "LegalActionRelationshipsId",
            sha2(concat_ws(";", *legactcom_primary_key_fields), 256),
        )


        # legalactiontoperson
        legalactiontoperson_primary_key_fields = [
            legalactiontoperson.RelKey,
            lit("LegalAction-Party"),
            lit("arotron"),
        ]

        legalactiontoperson = legalactiontoperson.withColumn(
            "LegalActionRelationshipsId",
            sha2(concat_ws(";", *legalactiontoperson_primary_key_fields), 256),
        )

        # ================== Selection of Final Columns ==================

        # result_df_legactrel
        result_df_legactrel = legactrel.select(
            legactrel.LegalActionRelationshipsId.alias("LegalActionRelationshipsId"),
            lit("arotron").cast("String").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            legactrel.RelationKey.cast("String").alias("LegalActionRelationshipsKey"),
            lit("LegalAction-LegalAction")
            .cast("String")
            .alias("LegalActionRelationshipsTypeCode"),
            lit("LegalAction-LegalAction")
            .cast("String")
            .alias("LegalActionRelationshipsType"),
            legactrel.RelationDescription.cast("String").alias(
                "LegalActionRelationshipsSubTypeCode"
            ),
            legactrel.RelationDescription.cast("String").alias(
                "LegalActionRelationshipsSubType"
            ),
            legactrel.PrimaryLegalSourceKey.cast("String").alias("LegalActionKey"),
            legactrel.SecondaryLegalSourceKey.cast("String").alias(
                "RelatedLegalActionKey"
            ),
            legactrel.RelCreationDate.cast("date").alias("CreationDate"),
            legactrel["RelTerminationDate"].cast("date").alias("TerminationDate"),
            lit(None).cast("date").alias("LastUpdateDate"),
            lit(None).cast("String").alias("Comments"),
            lit(None).cast("String").alias("PartyKey"),
            lit(None).cast("String").alias("PartyName"),
            legactrel["businessdate"].cast("date").alias("BusinessDate"),
        )

        # result_df_legactcom
        result_df_legactcom = legactcom.select(
            legactcom.LegalActionRelationshipsId.alias("LegalActionRelationshipsId"),
            lit("arotron").cast("String").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            legactcom.LegalActComId.cast("String").alias("LegalActionRelationshipsKey"),
            lit("LegalAction-Comments")
            .cast("String")
            .alias("LegalActionRelationshipsTypeCode"),
            lit("LegalAction-Comments")
            .cast("String")
            .alias("LegalActionRelationshipsType"),
            lit(None).cast("String").alias("LegalActionRelationshipsSubTypeCode"),
            lit(None).cast("String").alias("LegalActionRelationshipsSubType"),
            legactcom.LegalActId.cast("String").alias("LegalActionKey"),
            lit(None).cast("String").alias("RelatedLegalActionKey"),
            legactcom.CreateDT.cast("date").alias("CreationDate"),
            lit(None).cast("date").alias("TerminationDate"),
            legactcom.LastUpdDT.cast("date").alias("LastUpdateDate"),
            legactcom.Comments.cast("String").alias("Comments"),
            lit(None).cast("String").alias("PartyKey"),
            lit(None).cast("String").alias("PartyName"),
            legactcom["businessdate"].cast("date").alias("BusinessDate"),
        )

        # result_df_legalactiontoperson
        result_df_legalactiontoperson = legalactiontoperson.select(
            legalactiontoperson.LegalActionRelationshipsId.alias(
                "LegalActionRelationshipsId"
            ),
            lit("arotron").cast("String").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            legalactiontoperson.RelKey.cast("String").alias(
                "LegalActionRelationshipsKey"
            ),
            lit("LegalAction-Party")
            .cast("String")
            .alias("LegalActionRelationshipsTypeCode"),
            lit("LegalAction-Party")
            .cast("String")
            .alias("LegalActionRelationshipsType"),
            legalactiontoperson.RelationType.cast("String").alias(
                "LegalActionRelationshipsSubTypeCode"
            ),
            legalactiontoperson.RelationType.cast("String").alias(
                "LegalActionRelationshipsSubType"
            ),
            legalactiontoperson.LegalActId.cast("String").alias("LegalActionKey"),
            lit(None).cast("String").alias("RelatedLegalActionKey"),
            lit(None).cast("date").alias("CreationDate"),
            lit(None).cast("date").alias("TerminationDate"),
            lit(None).cast("date").alias("LastUpdateDate"),
            lit(None).cast("String").alias("Comments"),
            legalactiontoperson.CDI.cast("String").alias("PartyKey"),
            legalactiontoperson.AliasSecondary.cast("String").alias("PartyName"),
            legalactiontoperson["businessdate"].cast("date").alias("BusinessDate"),
        )

        # Union
        legal_action_relationships_arotron = result_df_legactrel.union(
            result_df_legactcom
        ).union(result_df_legalactiontoperson)

        # ================== Append ==================

        legal_action_relationships_arotron.write.format("delta").clusterBy(
            "BusinessDate"
        ).mode("append").saveAsTable(curated_table_path)

        # DONE till here

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                #destination_schema_zone, # change for testing
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_legal_action_relationships_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix) # change for testing